# SimpleTMG EDA: Why This Model Is Needed

This notebook is a full exploratory analysis for the benchmark datasets plus the cleaned `smartbuilding/smart.csv` dataset.

It is designed to answer these questions:

1. Why is a multivariate model needed instead of treating each channel independently?
2. Why is SWT useful for some datasets?
3. Why can FFT help on some datasets?
4. Why is a temporal-attention branch useful in addition to the original channel/scale mechanism?
5. Why does the cleaned smartbuilding dataset add value to the project?

The goal is not just to make plots. The goal is to connect measurable data properties to model design choices.

## What This Notebook Will Show

- dataset size, column structure, time frequency, missingness
- representative signal behavior across datasets
- cross-channel dependence to justify geometric/channel attention
- temporal persistence and seasonality to justify temporal attention
- spectral concentration to justify FFT tokenization
- multi-scale detail energy to justify SWT tokenization
- local variation and short-horizon behavior to explain when local patterns matter
- smartbuilding-specific structure: zone totals, floor total, environmental variables
- a final recommendation table mapping dataset properties to model choices


## How To Read The Results

Each section answers a concrete design question:

- **High cross-channel correlation**: a multivariate attention mechanism is justified.
- **Strong lag structure / repeated hourly profile**: a temporal branch is justified.
- **Low spectral entropy / clear dominant periods**: FFT has a good chance to help.
- **High wavelet detail energy / strong non-stationarity**: SWT has a good chance to help.
- **Smartbuilding coupling between power, temperature, lux, and zone totals**: this is a good stress-test for both tokenization and attention design.

This notebook is intentionally analysis-first. The metrics here are meant to support report writing and architectural justification, not only visualization.


In [ ]:
!pip install PyWavelets gdown statsmodels seaborn --quiet

import os
import zipfile
import warnings
from pathlib import Path

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pywt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, acf

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab10')
pd.set_option('display.max_columns', 120)

print('Libraries loaded successfully')


In [ ]:
FILE_ID = '1JOgmL2ntyxKg6eB132ytCbVZumiHnDLR'
OUTPUT_PATH = '/kaggle/working/dataset.zip'
DATASET_DIR = '/kaggle/working/dataset'



if not os.path.exists(DATASET_DIR):
    gdown.download(f'https://drive.google.com/uc?id={FILE_ID}', OUTPUT_PATH, quiet=False)
    os.makedirs(DATASET_DIR, exist_ok=True)
    with zipfile.ZipFile(OUTPUT_PATH, 'r') as zf:
        zf.extractall(DATASET_DIR)
    print('Dataset extracted')
else:
    print('Dataset already exists')

for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:8]:
        print(f'{subindent}{file}')


## Dataset Registry

The notebook loads the standard benchmark datasets and the cleaned smartbuilding dataset.

For the smartbuilding file, the analysis expects the cleaned hourly CSV with:
- `Date`
- zone-level power variables
- zone totals
- `Floor_Total(kW)`
- `Floor_Mean_Temp`
- `Floor_Mean_Lux`

This matches the current cleaned file produced from your preprocessing code.


In [ ]:
def resolve_existing_path(candidates):
    for candidate in candidates:
        if Path(candidate).exists():
            return str(candidate)
    return None

smart_candidates = [
    Path(DATASET_DIR) / 'SimpleTM_datasets' / 'smartbuilding' / 'smart.csv',
    Path(DATASET_DIR) / 'SimpleTM_datasets' / 'Smartbuilding' / 'smart.csv',
    Path('/kaggle/working/SimpleTM') / 'smart.csv',
    Path('smart.csv'),
]

DATASET_SPECS = [
    {'name': 'ETTh1', 'kind': 'csv', 'path': resolve_existing_path([Path(DATASET_DIR) / 'SimpleTM_datasets' / 'ETT-small' / 'ETTh1.csv'])},
    {'name': 'ETTh2', 'kind': 'csv', 'path': resolve_existing_path([Path(DATASET_DIR) / 'SimpleTM_datasets' / 'ETT-small' / 'ETTh2.csv'])},
    {'name': 'ETTm1', 'kind': 'csv', 'path': resolve_existing_path([Path(DATASET_DIR) / 'SimpleTM_datasets' / 'ETT-small' / 'ETTm1.csv'])},
    {'name': 'ETTm2', 'kind': 'csv', 'path': resolve_existing_path([Path(DATASET_DIR) / 'SimpleTM_datasets' / 'ETT-small' / 'ETTm2.csv'])},
    {'name': 'weather', 'kind': 'csv', 'path': resolve_existing_path([Path(DATASET_DIR) / 'SimpleTM_datasets' / 'weather' / 'weather.csv'])},
    {'name': 'electricity', 'kind': 'csv', 'path': resolve_existing_path([Path(DATASET_DIR) / 'SimpleTM_datasets' / 'electricity' / 'electricity.csv'])},
    {'name': 'traffic', 'kind': 'csv', 'path': resolve_existing_path([Path(DATASET_DIR) / 'SimpleTM_datasets' / 'traffic' / 'traffic.csv'])},
    {'name': 'Solar', 'kind': 'solar_txt', 'path': resolve_existing_path([Path(DATASET_DIR) / 'SimpleTM_datasets' / 'Solar' / 'solar_AL.txt'])},
    {'name': 'PEMS03', 'kind': 'pems_npz', 'path': resolve_existing_path([Path(DATASET_DIR) / 'SimpleTM_datasets' / 'PEMS' / 'PEMS03.npz'])},
    {'name': 'PEMS04', 'kind': 'pems_npz', 'path': resolve_existing_path([Path(DATASET_DIR) / 'SimpleTM_datasets' / 'PEMS' / 'PEMS04.npz'])},
    {'name': 'PEMS07', 'kind': 'pems_npz', 'path': resolve_existing_path([Path(DATASET_DIR) / 'SimpleTM_datasets' / 'PEMS' / 'PEMS07.npz'])},
    {'name': 'PEMS08', 'kind': 'pems_npz', 'path': resolve_existing_path([Path(DATASET_DIR) / 'SimpleTM_datasets' / 'PEMS' / 'PEMS08.npz'])},
    {'name': 'smartbuilding', 'kind': 'csv', 'path': resolve_existing_path(smart_candidates)},
]

DATASET_SPECS = [spec for spec in DATASET_SPECS if spec['path'] is not None]
print(pd.DataFrame(DATASET_SPECS).to_string(index=False))


In [ ]:
DATE_CANDIDATES = {'date', 'datetime', 'timestamp', 'time'}
EPS = 1e-8


def find_date_column(columns):
    for col in columns:
        if str(col).lower() in DATE_CANDIDATES:
            return col
    return None


def load_dataset(spec):
    path = Path(spec['path'])
    kind = spec['kind']
    if kind == 'csv':
        raw = pd.read_csv(path)
        date_col = find_date_column(raw.columns)
        if date_col is not None:
            raw[date_col] = pd.to_datetime(raw[date_col], errors='coerce')
            raw = raw.dropna(subset=[date_col]).set_index(date_col)
        numeric = raw.select_dtypes(include=[np.number]).copy()
    elif kind == 'solar_txt':
        raw = pd.read_csv(path, header=None)
        raw.columns = [f'feature_{i:03d}' for i in range(raw.shape[1])]
        numeric = raw
    elif kind == 'pems_npz':
        arr = np.load(path, allow_pickle=True)['data'][:, :, 0]
        numeric = pd.DataFrame(arr, columns=[f'node_{i:03d}' for i in range(arr.shape[1])])
    else:
        raise ValueError(kind)

    numeric = numeric.replace([np.inf, -np.inf], np.nan)
    return numeric


def safe_series(df):
    if 'Floor_Total(kW)' in df.columns:
        return df['Floor_Total(kW)']
    return df.iloc[:, 0]


def infer_frequency(index):
    if not isinstance(index, pd.DatetimeIndex) or len(index) < 4:
        return ''
    try:
        return pd.infer_freq(index[:min(len(index), 200)]) or ''
    except Exception:
        return ''


def classify_column(name):
    lower = str(name).lower()
    if 'total(kw)' in lower:
        return 'zone_or_floor_total_kw'
    if '(kw)' in lower:
        return 'power_kw'
    if 'degc' in lower or 'temp' in lower:
        return 'temperature'
    if 'rh%' in lower or 'humidity' in lower:
        return 'humidity'
    if 'lux' in lower:
        return 'light'
    return 'other'


def spectral_entropy(signal):
    signal = signal[np.isfinite(signal)]
    if signal.size < 8:
        return np.nan
    power = np.abs(np.fft.rfft(signal - signal.mean())) ** 2
    power = power[1:]
    if power.size == 0 or power.sum() <= EPS:
        return 0.0
    p = power / power.sum()
    return float(-(p * np.log(p + EPS)).sum() / np.log(len(p) + EPS))


def dominant_period(signal):
    signal = signal[np.isfinite(signal)]
    if signal.size < 8:
        return np.nan
    power = np.abs(np.fft.rfft(signal - signal.mean())) ** 2
    freq = np.fft.rfftfreq(len(signal), d=1.0)
    power[0] = 0.0
    idx = int(np.argmax(power))
    if idx <= 0 or freq[idx] <= 0:
        return np.nan
    return float(1.0 / freq[idx])


def adf_stationary_ratio(df, max_cols=12, sample_limit=5000):
    count = 0
    total = 0
    for col in df.columns[:max_cols]:
        values = df[col].dropna().values[:sample_limit]
        if len(values) < 50:
            continue
        total += 1
        try:
            p = adfuller(values, maxlag=min(24, len(values) // 4))[1]
            if p < 0.05:
                count += 1
        except Exception:
            pass
    return np.nan if total == 0 else 100.0 * count / total


def wavelet_detail_ratio(signal, wavelet='db1', level=3):
    signal = signal[np.isfinite(signal)]
    if signal.size < 64:
        return np.nan
    pad_len = int(2 ** np.ceil(np.log2(len(signal))))
    padded = np.pad(signal, (0, pad_len - len(signal)), mode='reflect')
    coeffs = pywt.swt(padded.astype(float), wavelet, level=level)
    approx_energy = np.sum(coeffs[-1][0] ** 2)
    detail_energy = sum(np.sum(detail ** 2) for _, detail in coeffs)
    total = approx_energy + detail_energy + EPS
    return float(100.0 * detail_energy / total)


def autocorr_half_life(signal, max_lag=72):
    signal = signal[np.isfinite(signal)]
    if signal.size < max_lag + 5:
        return np.nan
    values = acf(signal, nlags=max_lag, fft=True)
    for lag in range(1, len(values)):
        if values[lag] < 0.5:
            return float(lag)
    return float(max_lag)


def mean_abs_corr(df, max_cols=20):
    subset = df.iloc[:, :min(max_cols, df.shape[1])]
    corr = subset.corr().values
    if corr.size == 0:
        return np.nan
    mask = ~np.eye(corr.shape[0], dtype=bool)
    vals = np.abs(corr[mask])
    vals = vals[np.isfinite(vals)]
    return float(vals.mean()) if vals.size else np.nan


def weekday_hour_profile_strength(df):
    if not isinstance(df.index, pd.DatetimeIndex):
        return np.nan
    series = safe_series(df)
    if len(series) < 48:
        return np.nan
    profile = series.groupby(df.index.hour).mean()
    return float(profile.std() / (series.std() + EPS))


def local_variance_ratio(signal, window=12):
    signal = signal[np.isfinite(signal)]
    if signal.size < window * 2:
        return np.nan
    local_var = pd.Series(signal).rolling(window).var().dropna()
    return float(local_var.mean() / (np.var(signal) + EPS))


def compute_summary(name, df, path, kind):
    series = safe_series(df).dropna().values[:5000]
    return {
        'Dataset': name,
        'Kind': kind,
        'Path': path,
        'Rows': len(df),
        'Features': df.shape[1],
        'Missing': int(df.isna().sum().sum()),
        'ZeroRatio': float((df == 0).sum().sum() / max(df.size, 1)),
        'Start': df.index.min() if isinstance(df.index, pd.DatetimeIndex) else '',
        'End': df.index.max() if isinstance(df.index, pd.DatetimeIndex) else '',
        'Freq': infer_frequency(df.index),
        'RepresentativeSeries': safe_series(df).name,
        'StationaryPct': adf_stationary_ratio(df),
        'MeanAbsCorr': mean_abs_corr(df),
        'TemporalProfileStrength': weekday_hour_profile_strength(df),
        'AutocorrHalfLife': autocorr_half_life(series),
        'SpectralEntropy': spectral_entropy(series),
        'DominantPeriod': dominant_period(series),
        'WaveletDetailPct': wavelet_detail_ratio(series),
        'LocalVarianceRatio': local_variance_ratio(series),
    }


datasets = {spec['name']: load_dataset(spec) for spec in DATASET_SPECS}
summary_df = pd.DataFrame([
    compute_summary(spec['name'], datasets[spec['name']], spec['path'], spec['kind'])
    for spec in DATASET_SPECS
]).sort_values('Dataset').reset_index(drop=True)
summary_df


## 1. Dataset Overview

This section answers:

- How big is each dataset?
- How many channels does it have?
- Does it have a clean timestamp?
- Does it have lots of zeros or missing values?
- Is the smartbuilding dataset structurally different from the standard benchmarks?

Why this matters:

- The model has to work across very different channel counts and time structures.
- High-dimensional datasets motivate the inverted embedding and channel-aware attention.
- Smartbuilding adds semantically grouped features: equipment loads, zone totals, temperature, humidity, and lux.


In [ ]:
summary_df


In [ ]:
smart_df = datasets['smartbuilding'] if 'smartbuilding' in datasets else None
if smart_df is not None:
    smart_columns = pd.DataFrame({
        'Column': smart_df.columns,
        'Role': [classify_column(c) for c in smart_df.columns],
        'DType': [str(smart_df[c].dtype) for c in smart_df.columns],
    })
    display(smart_columns)
else:
    print('smartbuilding dataset not found')


## 2. Representative Series

This section answers:

- What do the raw signals look like?
- Are the series smooth, noisy, bursty, trending, or strongly periodic?
- Is the smartbuilding total load visibly different from the benchmark signals?

Why this matters:

- If the signal is strongly periodic and smooth, FFT may help.
- If the signal has abrupt local changes or changing regimes, SWT becomes more important.
- If there are recurring daily operational cycles, temporal modeling is justified.


In [ ]:
n = len(datasets)
fig, axes = plt.subplots(n, 1, figsize=(14, 3 * n), squeeze=False)
axes = axes.flatten()
for ax, (name, df) in zip(axes, datasets.items()):
    series = safe_series(df).iloc[:1000]
    x = series.index if isinstance(series.index, pd.DatetimeIndex) else np.arange(len(series))
    ax.plot(x, series.values, linewidth=0.9)
    ax.set_title(f'{name} - {series.name}')
    ax.set_ylabel('value')
axes[-1].set_xlabel('time')
plt.tight_layout()
plt.show()


## 3. Cross-Channel Dependence: Why A Multivariate Model Is Needed

Question:

- Can we justify channel-aware modeling, or would a purely univariate approach be enough?

What we measure:

- mean absolute cross-channel correlation
- per-dataset correlation heatmaps on a feature subset

Why it matters:

- Strong cross-channel dependence supports the original SimpleTM geometric attention branch.
- The smartbuilding dataset should be especially useful here because equipment loads, zone totals, and floor aggregates are physically coupled.


In [ ]:
corr_view = summary_df[['Dataset', 'Features', 'MeanAbsCorr']].sort_values('MeanAbsCorr', ascending=False)
corr_view


In [ ]:
selected = list(datasets.items())[:6]
fig, axes = plt.subplots(len(selected), 1, figsize=(11, 4 * len(selected)), squeeze=False)
axes = axes.flatten()
for ax, (name, df) in zip(axes, selected):
    subset = df.iloc[:, :min(12, df.shape[1])]
    corr = subset.corr()
    sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0, cbar=False)
    ax.set_title(f'{name} - correlation heatmap (first {subset.shape[1]} channels)')
plt.tight_layout()
plt.show()


## 4. Temporal Dependence: Why A Temporal-Attention Branch Is Reasonable

Question:

- Do the datasets contain enough temporal persistence and repeated profiles to justify adding a temporal branch in parallel to the original channel/scale mechanism?

What we measure:

- autocorrelation half-life
- hour-of-day profile strength for timestamped datasets

Why it matters:

- Long persistence and repeated daily profiles mean the model can benefit from directly reasoning over the temporal axis.
- This is the main empirical justification for the dual-attention extension.


In [ ]:
temporal_view = summary_df[['Dataset', 'AutocorrHalfLife', 'TemporalProfileStrength', 'StationaryPct']].sort_values('AutocorrHalfLife', ascending=False)
temporal_view


In [ ]:
timestamped = [(name, df) for name, df in datasets.items() if isinstance(df.index, pd.DatetimeIndex)]
fig, axes = plt.subplots(len(timestamped), 1, figsize=(10, 3 * len(timestamped)), squeeze=False)
axes = axes.flatten()
for ax, (name, df) in zip(axes, timestamped):
    hourly = safe_series(df).groupby(df.index.hour).mean()
    ax.plot(hourly.index, hourly.values, marker='o')
    ax.set_title(f'{name} - mean hour-of-day profile')
    ax.set_xlabel('hour')
    ax.set_ylabel(safe_series(df).name)
plt.tight_layout()
plt.show()


## 5. Frequency-Domain Structure: Why FFT Can Help

Question:

- Which datasets have compact spectral structure and strong global periodicity?

What we measure:

- spectral entropy
- dominant period
- power spectrum plots

Interpretation:

- **Lower spectral entropy** means energy is concentrated in a smaller number of frequencies.
- **Clear dominant periods** mean FFT tokenization has a strong structural prior to exploit.

Expected outcome:

- Solar and some operational datasets often look more FFT-friendly than irregular load/noise-heavy datasets.


In [ ]:
fft_view = summary_df[['Dataset', 'SpectralEntropy', 'DominantPeriod']].sort_values('SpectralEntropy')
fft_view


In [ ]:
selected = list(datasets.items())[:8]
fig, axes = plt.subplots(len(selected), 1, figsize=(12, 3 * len(selected)), squeeze=False)
axes = axes.flatten()
for ax, (name, df) in zip(axes, selected):
    series = safe_series(df).dropna().values[:5000]
    centered = series - series.mean()
    power = np.abs(np.fft.rfft(centered)) ** 2
    freq = np.fft.rfftfreq(len(centered), d=1.0)
    upper = max(8, len(freq) // 4)
    ax.semilogy(freq[1:upper], power[1:upper] + EPS)
    ax.set_title(f'{name} - power spectrum')
    ax.set_xlabel('frequency')
    ax.set_ylabel('power')
plt.tight_layout()
plt.show()


## 6. Multi-Scale / Non-Stationary Structure: Why SWT Can Help

Question:

- Which datasets have changing regimes, transient behavior, and non-stationary multi-scale structure?

What we measure:

- ADF-based stationary ratio
- wavelet detail energy percentage

Interpretation:

- **Lower stationary ratio** means the data are less stable over time.
- **Higher wavelet detail percentage** means there is substantial fine-scale and transient structure.

Why it matters:

- SWT keeps temporal locality and separates coarse trend from multi-scale detail.
- This is the most direct empirical justification for retaining SWT in the study.


In [ ]:
swt_view = summary_df[['Dataset', 'StationaryPct', 'WaveletDetailPct']].sort_values('WaveletDetailPct', ascending=False)
swt_view


In [ ]:
selected = list(datasets.items())[:6]
fig, axes = plt.subplots(len(selected), 4, figsize=(16, 3 * len(selected)), squeeze=False)
for row_idx, (name, df) in enumerate(selected):
    signal = safe_series(df).dropna().values[:256].astype(float)
    signal = signal - signal.mean()
    pad_len = int(2 ** np.ceil(np.log2(len(signal))))
    padded = np.pad(signal, (0, pad_len - len(signal)), mode='reflect')
    coeffs = pywt.swt(padded, 'db1', level=3)

    axes[row_idx, 0].plot(signal, linewidth=0.8)
    axes[row_idx, 0].set_title(f'{name} - original')
    axes[row_idx, 1].plot(coeffs[-1][0][:len(signal)], color='red', linewidth=0.8)
    axes[row_idx, 1].set_title('approximation')
    axes[row_idx, 2].plot(coeffs[-1][1][:len(signal)], color='green', linewidth=0.8)
    axes[row_idx, 2].set_title('detail low')
    axes[row_idx, 3].plot(coeffs[0][1][:len(signal)], color='blue', linewidth=0.8)
    axes[row_idx, 3].set_title('detail high')
plt.tight_layout()
plt.show()


## 7. Local Pattern Strength

Question:

- Do some datasets contain strong short-range changes and local variation?

What we measure:

- autocorrelation half-life
- local variance ratio

Why it matters:

- Even when the main comparison is SWT vs FFT, local behavior still matters because it explains why some datasets are harder and why a purely global frequency view may not always be enough.


In [ ]:
local_view = summary_df[['Dataset', 'AutocorrHalfLife', 'LocalVarianceRatio']].sort_values('LocalVarianceRatio', ascending=False)
local_view


## 8. Smartbuilding Deep Dive

Question:

- What exactly makes the smartbuilding dataset interesting for this project?

Why this dataset matters:

- It has semantically meaningful groups of variables rather than anonymous channels.
- Zone loads, totals, temperature, humidity, and lux are physically linked.
- It gives you a concrete building-systems forecasting case instead of only benchmark competition datasets.
- It is useful for discussing both forecasting and interpretability in the report.

The following plots focus only on smartbuilding.


In [ ]:
if smart_df is not None:
    zone_total_cols = [c for c in smart_df.columns if c.endswith('_Total(kW)') and c != 'Floor_Total(kW)']
    display(pd.DataFrame({
        'Column': smart_df.columns,
        'Role': [classify_column(c) for c in smart_df.columns],
    }))

    fig, ax = plt.subplots(figsize=(14, 5))
    zoom = smart_df[zone_total_cols + ['Floor_Total(kW)']].iloc[:24*14]
    for col in zone_total_cols:
        ax.plot(zoom.index, zoom[col], linewidth=1.0, alpha=0.8, label=col)
    ax.plot(zoom.index, zoom['Floor_Total(kW)'], linewidth=2.2, color='black', label='Floor_Total(kW)')
    ax.set_title('smartbuilding - zone totals and floor total')
    ax.legend(ncol=2, fontsize=8)
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(smart_df['Floor_Mean_Temp'], smart_df['Floor_Total(kW)'], alpha=0.35, s=10)
    axes[0].set_title('floor total vs mean temperature')
    axes[0].set_xlabel('Floor_Mean_Temp')
    axes[0].set_ylabel('Floor_Total(kW)')

    axes[1].scatter(smart_df['Floor_Mean_Lux'], smart_df['Floor_Total(kW)'], alpha=0.35, s=10)
    axes[1].set_title('floor total vs mean lux')
    axes[1].set_xlabel('Floor_Mean_Lux')
    axes[1].set_ylabel('Floor_Total(kW)')
    plt.tight_layout()
    plt.show()

    hourly_heatmap = (
        smart_df.assign(hour=smart_df.index.hour, weekday=smart_df.index.day_name())
        .pivot_table(index='weekday', columns='hour', values='Floor_Total(kW)', aggfunc='mean')
        .reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
    )
    plt.figure(figsize=(12, 4))
    sns.heatmap(hourly_heatmap, cmap='viridis')
    plt.title('smartbuilding - mean floor load by weekday and hour')
    plt.tight_layout()
    plt.show()
else:
    print('smartbuilding dataset not found')


## 9. Final Interpretation Table

This table is the main report-facing summary.

Read it like this:

- **ChannelNeed**: how strongly the data justify cross-channel modeling
- **TemporalNeed**: how strongly the data justify a temporal branch
- **FFTNeed**: how favorable the data look for FFT tokenization
- **SWTNeed**: how favorable the data look for SWT tokenization

These labels are heuristic. They are not a substitute for experiments, but they make the architecture rationale explicit.


In [ ]:
final_df = summary_df.copy()

final_df['ChannelNeed'] = pd.cut(
    final_df['MeanAbsCorr'],
    bins=[-np.inf, 0.1, 0.25, np.inf],
    labels=['Low', 'Medium', 'High']
)
final_df['TemporalNeed'] = pd.cut(
    final_df['AutocorrHalfLife'],
    bins=[-np.inf, 8, 24, np.inf],
    labels=['Low', 'Medium', 'High']
)
final_df['FFTNeed'] = pd.cut(
    final_df['SpectralEntropy'],
    bins=[-np.inf, 0.45, 0.65, np.inf],
    labels=['High', 'Medium', 'Low']
)
final_df['SWTNeed'] = pd.cut(
    final_df['WaveletDetailPct'],
    bins=[-np.inf, 15, 30, np.inf],
    labels=['Low', 'Medium', 'High']
)

final_cols = [
    'Dataset', 'Features', 'MeanAbsCorr', 'AutocorrHalfLife', 'SpectralEntropy',
    'WaveletDetailPct', 'ChannelNeed', 'TemporalNeed', 'FFTNeed', 'SWTNeed'
]
final_df[final_cols].sort_values('Dataset')


## 10. Report-Style Conclusions

Use the notebook outputs to support conclusions such as:

1. The benchmark set is heterogeneous in both channel count and temporal structure, so a single simplistic temporal encoder is not enough.
2. Cross-channel dependence is substantial in several datasets, which supports the original SimpleTM channel/scale attention mechanism.
3. Strong temporal persistence and repeated daily profiles support the addition of a parallel temporal-attention branch.
4. Some datasets show concentrated spectral structure and clear dominant periods, which supports testing FFT tokenization.
5. Other datasets show stronger non-stationary and multi-scale detail behavior, which supports SWT tokenization.
6. The smartbuilding dataset is especially useful because it adds a physically interpretable multivariate setting with equipment, environmental, and aggregate variables.

This is the bridge between raw EDA and the model architecture section of the report.
